# Detection and Tracking: People Walking

This notebook detects people in a video with [RF-DETR](https://github.com/roboflow/rf-detr) and gives each person a persistent ID across frames with ByteTrack from Roboflow's [trackers](https://github.com/roboflow/trackers) package. [Supervision](https://supervision.roboflow.com/latest) handles reading and writing the video and drawing the annotations.

1. Download the sample video.
2. Detect people on a single frame.
3. Track people across the whole video and save the annotated result.

## Setup

In [ ]:
from pathlib import Path

import supervision as sv
import torch
from rfdetr import RFDETRNano
from rfdetr.assets.coco_classes import COCO_CLASSES
from supervision.assets import VideoAssets, download_assets
from trackers import ByteTrackTracker

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("supervision", sv.__version__)
print("MPS available:", torch.backends.mps.is_available())

## Download the video

Supervision ships a few sample videos. `PEOPLE_WALKING` shows a crowd of pedestrians filmed from above.

In [ ]:
SOURCE_VIDEO_PATH = download_assets(VideoAssets.PEOPLE_WALKING, directory=DATA_DIR)
TARGET_VIDEO_PATH = OUTPUT_DIR / "people_walking_tracked.mp4"

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
video_info

In [ ]:
frame = next(sv.get_video_frames_generator(SOURCE_VIDEO_PATH))
sv.plot_image(frame, size=(12, 8))

## Load the model

RF-DETR comes in several sizes (`RFDETRNano`, `RFDETRSmall`, `RFDETRMedium`, `RFDETRBase`, `RFDETRLarge`), all pretrained on COCO. Nano is the fastest; swap in a bigger one for better accuracy. Weights are downloaded to `~/.roboflow/models/` on first use.

In [ ]:
model = RFDETRNano()

## Detect people on a single frame

`model.predict` returns an `sv.Detections` object. Supervision reads frames in BGR order (OpenCV convention) while RF-DETR expects RGB, so we flip the channels with `frame[:, :, ::-1]`.

We keep only the `person` class. Class IDs follow the COCO convention, so `COCO_CLASSES` maps them to names.

In [ ]:
PERSON_CLASS_ID = next(k for k, v in COCO_CLASSES.items() if v == "person")

detections = model.predict(frame[:, :, ::-1].copy(), threshold=0.5)
detections = detections[detections.class_id == PERSON_CLASS_ID]
print(f"{len(detections)} people detected")

In [ ]:
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator(smart_position=True)

labels = [
    f"{COCO_CLASSES[class_id]} {confidence:.2f}"
    for class_id, confidence in zip(detections.class_id, detections.confidence)
]

annotated_frame = box_annotator.annotate(frame.copy(), detections)
annotated_frame = label_annotator.annotate(annotated_frame, detections, labels)
sv.plot_image(annotated_frame, size=(12, 8))

### Small objects: slice the frame

Only a fraction of the people are detected. The model resizes each 1920×1080 frame down to its input resolution (384 px for Nano), so people far from the camera shrink to a handful of pixels. A bigger RF-DETR variant doesn't help much here; the input resolution is the bottleneck.

`sv.InferenceSlicer` fixes this by splitting the frame into overlapping 640×640 tiles, running the model on each tile at full detail, and merging the results with non-max suppression (the idea behind [SAHI](https://github.com/obss/sahi)). It's slower (one model call per tile) but finds far more people.

In [ ]:
def detect(image):
    return model.predict(image[:, :, ::-1].copy(), threshold=0.3)


slicer = sv.InferenceSlicer(callback=detect, slice_wh=640, overlap_wh=128)

detections = slicer(frame)
detections = detections[(detections.class_id == PERSON_CLASS_ID) & (detections.confidence >= 0.5)]
print(f"{len(detections)} people detected")

annotated_frame = box_annotator.annotate(frame.copy(), detections)
sv.plot_image(annotated_frame, size=(12, 8))

## Track people across the video

A detector only looks at one frame at a time, so the same person gets no identity between frames. A tracker links detections over time and assigns each object a `tracker_id`.

ByteTrack matches high-confidence detections first and then uses low-confidence ones to keep tracks alive through partial occlusions. That's why the slicer keeps detections down to 0.3 confidence and lets the tracker decide what becomes a track.

Colouring by `ColorLookup.TRACK` gives each ID its own colour, and `TraceAnnotator` draws the path each person has walked.

In [ ]:
tracker = ByteTrackTracker(frame_rate=video_info.fps)

box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK, smart_position=True)
trace_annotator = sv.TraceAnnotator(color_lookup=sv.ColorLookup.TRACK, trace_length=60)


def callback(frame, index):
    detections = slicer(frame)
    detections = detections[detections.class_id == PERSON_CLASS_ID]
    detections = tracker.update(detections)
    detections = detections[detections.tracker_id != -1]

    labels = [f"#{tracker_id}" for tracker_id in detections.tracker_id]

    annotated = trace_annotator.annotate(frame.copy(), detections)
    annotated = box_annotator.annotate(annotated, detections)
    annotated = label_annotator.annotate(annotated, detections, labels)
    return annotated

`sv.process_video` reads every frame, runs the callback and writes the result. Set `max_frames` to a small number for a quick test run; the full video (341 frames) takes about a minute and a half on an Apple Silicon Mac.

In [ ]:
sv.process_video(
    source_path=str(SOURCE_VIDEO_PATH),
    target_path=str(TARGET_VIDEO_PATH),
    callback=callback,
    max_frames=None,
    show_progress=True,
)

## Inspect the result

A few frames sampled from the output video. The same person should keep the same ID and colour throughout. Open `output/people_walking_tracked.mp4` to watch the full video.

In [ ]:
frames = list(sv.get_video_frames_generator(TARGET_VIDEO_PATH, stride=video_info.total_frames // 5))[1:5]
sv.plot_images_grid(frames, grid_size=(2, 2), size=(16, 9))